# ActivIQ revised-prompt and LoRA rerun

Use a fresh Colab A100 runtime. Run through the prompt comparison first.
Read both prompts and set PROMPT_FROZEN to True in the setup cell.
The notebook pauses before training. Inspect the scores, then set
APPROVE_NEW_LABELS_AND_TRAINING to True in the decision cell if you want to continue.

The reference is the saved winning APE instruction, scored again under the current
environment. This run also generates one common candidate so both comparisons
use the existing workflow's model and seed provenance checks.
All prompt selection uses the same 100 validation rows.
New results go to MyDrive/ActivIQ_prompt_v2.


In [ ]:
# 1. Config + Drive + repo (clones once; never deletes an existing checkout).
import os, subprocess, sys, time
T0 = time.time()
def elapsed(): print(f"[elapsed {(time.time()-T0)/60:.1f} min]")
def run_script(script, *args):
    subprocess.run([sys.executable, f"scripts/{script}.py", *args], check=True)

PROMPT_FROZEN = False   # <-- set True after reading the saved winning instruction and precision_challenger.txt

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ActivIQ_prompt_v2"
os.makedirs(DRIVE, exist_ok=True)

REPO, BRANCH = "jiakai2002/ActivIQ", "main"
# Private repo? Add GITHUB_TOKEN in Colab Secrets (key icon), then uncomment:
# from google.colab import userdata; os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
tok = os.environ.get("GITHUB_TOKEN")
url = f"https://{tok+'@' if tok else ''}github.com/{REPO}.git"
if not os.path.exists("/content/ActivIQ"):
    !git clone -q --depth 1 -b {BRANCH} {url} /content/ActivIQ
%cd /content/ActivIQ
if not os.path.lexists("reports"):
    os.symlink(DRIVE, "reports")
if not os.path.islink("reports") or os.path.realpath("reports") != DRIVE:
    raise RuntimeError("Existing reports path belongs to another run. Start a fresh Colab runtime.")
!git log --oneline -1
elapsed()

In [ ]:
# 2. Dependencies (keep Colab's torch) + pick settings from the GPU.
# flash-linear-attention = fast kernels for Qwen3.5's linear-attention layers. causal-conv1d is skipped on purpose
# (it compiles from source and often hangs on Colab; the PyTorch fallback for it is cheap).
!pip install -q "transformers>=5.3,<6" "peft>=0.13,<1" "accelerate>=1,<2" safetensors bitsandbytes pandas spacy flash-linear-attention
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type."
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
# 9B in bf16 needs ~19 GB: use it when the GPU has room (A100). Otherwise 4-bit.
Q9B = "none" if mem >= 30 else "4bit"
# 2B: full precision on bf16 GPUs, 4-bit on a T4.
Q2B = "none" if BF16 else "4bit"
print(torch.__version__, gpu, f"{mem:.0f} GB")
print("9B quantization:", Q9B, "| 2B quantization:", Q2B)
if "T4" in gpu or "L4" in gpu:
    print("WARNING: this GPU is slow for the 9B stages (estimate: many hours). An A100 is much faster.")
elapsed()

In [ ]:
# Inspect reference data and both prompts before loading a checkpoint.
from pathlib import Path
import json
run_script("auto_label", "check")
reference = json.loads(Path("ActivIQ_runs/auto_labelling/score/selection.json").read_text())
OLD_PROMPT = reference["selected_prompt"]["instruction"]
CHALLENGER_PATH = "prompts/auto_labelling/precision_challenger.txt"
print("SAVED WINNING PROMPT", reference["selected_prompt"]["id"])
print(OLD_PROMPT)
print("\nCHALLENGER PROMPT")
print(Path(CHALLENGER_PATH).read_text())
Path("reports/reference_prompt.txt").write_text(OLD_PROMPT, encoding="utf-8")
assert PROMPT_FROZEN, "Review both prompts, set PROMPT_FROZEN=True in setup, and rerun."


In [ ]:
# 4. Download the pinned checkpoints into the local Hugging Face cache (~19 GB + ~5 GB).
import json
from pathlib import Path
from huggingface_hub import snapshot_download
revs = json.loads(Path("ActivIQ_runs/model_revisions.json").read_text(encoding="utf-8"))
QWEN_9B = snapshot_download("Qwen/Qwen3.5-9B", revision=revs["Qwen/Qwen3.5-9B"])
Path(DRIVE, "model_revisions.json").write_text(json.dumps(revs, indent=2), encoding="utf-8")
print(revs)
elapsed()

In [ ]:
# 5. APE step 1: the 9B model proposes candidate prompts from the 10 seeds.
run_script("auto_label", "propose", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
           "--candidate-count", "1", "--run-dir", "reports/auto_labelling/propose", "--resume")
elapsed()

In [ ]:
# Rescore the original winner and the challenger in the same current environment.
for run_name, prompt_path in (
    ("reference", "reports/reference_prompt.txt"),
    ("challenger", CHALLENGER_PATH),
):
    run_script("auto_label", "score", "--model-path", QWEN_9B,
               "--device", "cuda", "--quantization", Q9B,
               "--candidates", "reports/auto_labelling/propose/candidates.jsonl",
               "--baseline", prompt_path,
               "--run-dir", f"reports/auto_labelling/score_{run_name}", "--resume")
elapsed()


In [ ]:
# Each run calls its supplied instruction "baseline".
import pandas as pd
REFERENCE_SELECTION = "reports/auto_labelling/score_reference/selection.json"
CHALLENGER_SELECTION = "reports/auto_labelling/score_challenger/selection.json"
comparison = []
for name, path in (("original winning instruction", REFERENCE_SELECTION),
                   ("precision challenger", CHALLENGER_SELECTION)):
    report = json.loads(Path(path).read_text())
    metrics = report["baseline"]["metrics"]
    comparison.append({"instruction": name, "entity_micro_f1": metrics["micro"]["f1"],
                       "valid_output_rate": metrics["valid_output_rate"],
                       "intent_agreement": metrics["intent_agreement"]})
display(pd.DataFrame(comparison))
print("Historical original winner F1 was 0.700. Compare the rescored reference above.")
print("Scores select prompts on validation. The 0.80 auto-labelling target needs all-slot coverage.")


In [ ]:
# Review the score table before choosing to spend time on labels and training.
APPROVE_NEW_LABELS_AND_TRAINING = False
reference_score = comparison[0]["entity_micro_f1"]
selections = [(REFERENCE_SELECTION, json.loads(Path(REFERENCE_SELECTION).read_text())),
              (CHALLENGER_SELECTION, json.loads(Path(CHALLENGER_SELECTION).read_text()))]
SELECTED_PATH, selected = max(
    selections, key=lambda pair: pair[1]["metrics"]["micro"]["f1"]
)
new_score = selected["metrics"]["micro"]["f1"]
print("Selected run:", SELECTED_PATH)
print("Selected prompt:", selected["selected_prompt"]["id"], "F1:", new_score)
print("Full schema-valid output rate:", selected["metrics"]["valid_output_rate"])
assert new_score > reference_score, "No prompt beat the rescored original instruction. Keep the existing labels and adapter."
assert selected["metrics"]["valid_output_rate"] == 1.0, "Inspect rejected validation outputs before proceeding."
assert APPROVE_NEW_LABELS_AND_TRAINING, "Review results, then set approval True in this cell and rerun this cell onward."


In [ ]:
# 8. Label the 960 training utterances with the selected prompt (the long step; safe to resume).
run_script("auto_label", "label", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
           "--selection", SELECTED_PATH,
           "--run-dir", "reports/auto_labelling/train_labels", "--resume")
elapsed()

In [ ]:
# Inspect the training label summary. Invalid rows count in its denominator.
summary = json.loads(Path("reports/auto_labelling/train_labels/summary.json").read_text())
display(summary)
assert summary["accepted"] == 960 and summary["rejected"] == 0, "Review label failures before training."
assert summary["intent_agreement_target_met"], "Intent agreement fell below 0.85. Review labels."


In [ ]:
# Load the smaller checkpoint only after the labelling quality check.
QWEN_2B = snapshot_download("Qwen/Qwen3.5-2B", revision=revs["Qwen/Qwen3.5-2B"])


In [ ]:
!pip uninstall -y torchao

In [ ]:
# 10. LoRA fine-tune Qwen3.5-2B on the machine labels (skipped if the adapter already exists on Drive).
if os.path.exists("reports/lora/adapter_v1/training_report.json"):
    if not os.path.isfile("reports/lora/adapter_v1/adapter_model.safetensors"):
        raise FileNotFoundError("The training report exists, but the adapter weights are missing. Restore them or use a new run directory.")
    print("adapter_v1 already trained, skipping")
else:
    run_script("train_lora", "--model-path", QWEN_2B, "--device", "cuda", "--quantization", Q2B,
               "--labels", "reports/auto_labelling/train_labels/machine_labels.jsonl",
               "--output-dir", "reports/lora/adapter_v1")
elapsed()

## Baseline checkpoint

Optional. Put intent_classifier.zip in MyDrive/ActivIQ_prompt_v2 to restore your
original DistilBERT model. Otherwise the following cell retrains it using the
existing recipe. The baseline extractor is the version in this checkout.
Report the project revision and model identity alongside the comparison.


In [ ]:
# 11. Restore DistilBERT if you uploaded it.
z = f"{DRIVE}/intent_classifier.zip"
HAVE_BASELINE = os.path.exists(z)
if HAVE_BASELINE:
    !mkdir -p models && unzip -qo {z} -d models/
    !ls models/intent_classifier | head
print("baseline available:", HAVE_BASELINE)

In [ ]:
# 11b. No DistilBERT zip? Retrain it with the repo's own script (same recipe, new weights).
!if [ ! -f models/intent_classifier/config.json ]; then mkdir -p .tmp && sed 's/^\( *\)tokenizer=tokenizer,$/\1processing_class=tokenizer,/' scripts/train_intent.py > .tmp/train_intent_colab.py && python .tmp/train_intent_colab.py; fi
HAVE_BASELINE = os.path.exists("models/intent_classifier/config.json"); print("baseline available:", HAVE_BASELINE)

In [ ]:
# 12. Evaluate LoRA 2B (and DistilBERT + spaCy if available) on the 120 test utterances.
skip = [] if HAVE_BASELINE else ["--skip-baseline"]
run_script("evaluate_lora", "--model-path", QWEN_2B, "--adapter", "reports/lora/adapter_v1",
           "--device", "cuda", "--quantization", Q2B, *skip,
           "--output-dir", "reports/evaluation/lora_2b")
elapsed()

In [ ]:
# 13. Final comparison table.
def row(name, r):
    e = r["entities"]; f1 = e["micro"]["f1"] if "micro" in e else e.get("status")
    return {"system": name, "intent_acc": r["intent"]["accuracy"], "intent_macro_f1": r["intent"]["macro_f1"],
            "entity_micro_f1": f1, "median_latency_s": r["latency_seconds"].get("median_seconds"),
            "invalid_outputs": r.get("invalid_outputs", 0)}
rep = json.load(open("reports/evaluation/lora_2b/lora_comparison.json"))
table = []
if "distilbert_spacy" in rep: table.append(row("DistilBERT + spaCy", rep["distilbert_spacy"]))
table.append(row("Qwen3.5-2B + LoRA (machine labels)", rep["lora"]))
zs = "reports/evaluation/qwen9b_zeroshot/lora_comparison.json"
if os.path.exists(zs): table.append(row("Qwen3.5-9B zero-shot", json.load(open(zs))["lora"]))
pd.set_option("display.float_format", "{:.3f}".format)
display(pd.DataFrame(table))
print("Latency: GPU for Qwen, CPU/GPU per your runtime; state the hardware in the report.")
print("Report caveats: seeds/validation labels were AI-drafted then human-reviewed; validation scores are selection scores, not held-out;",
      "test entity F1 is pending unless entities_test.jsonl is human-labelled; model revisions are in model_revisions.json on Drive.")
elapsed()

In [ ]:
# Save prompt tables, summaries and evaluation outputs for the report.
# The complete adapter and training labels remain in the new Drive folder.
import zipfile
with zipfile.ZipFile("revised_prompt_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ("auto_labelling/score_reference", "auto_labelling/score_challenger", "evaluation"):
        for path in Path("reports", folder).rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to("reports"))
    for name in ("auto_labelling/train_labels/summary.json",
                 "lora/adapter_v1/training_report.json", "model_revisions.json"):
        path = Path("reports", name)
        if path.is_file():
            archive.write(path, name)
from google.colab import files
files.download("revised_prompt_results.zip")
